In [1]:
import sys
import copy
import json
import math
import shutil
import hashlib
import zipfile
import tempfile
import subprocess
import importlib
import importlib.util
import importlib.metadata

from collections import Counter
from pathlib import Path


# ============================================================
# 0. CONFIG
# ============================================================
BASELINE_DIR = Path(
    "/kaggle/input/models/hoangvux/baselinev1/other/default/16"
)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/neurogolf-2026"
)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

WORK_DIR = (
    KAGGLE_WORKING
    / "next_task_from_715132"
)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONTROL_ZIP = (
    KAGGLE_WORKING
    / "submission_control_715132.zip"
)

TEST_ZIP = (
    KAGGLE_WORKING
    / "submission_next_task_test.zip"
)

FINAL_ZIP = (
    KAGGLE_WORKING
    / "submission.zip"
)

SCAN_CSV = (
    KAGGLE_WORKING
    / "next_task_scan_715132.csv"
)

EVALUATION_CSV = (
    KAGGLE_WORKING
    / "next_task_evaluation_715132.csv"
)

REPORT_JSON = (
    KAGGLE_WORKING
    / "next_task_report_715132.json"
)

EXPECTED_TASKS = {
    f"task{i:03d}.onnx"
    for i in range(1, 401)
}

# task352 đã tạo ra baseline 7151.32.
# Không sửa lại trong vòng này.
EXCLUDED_TASKS = {
    # "task005.onnx",
    # "task185.onnx",
    # "task046.onnx",
    # "task132.onnx",
    # "task201.onnx"
}

# Số task tiềm năng lớn nhất được chạy
# public validation và official scorer.
MAX_TASKS_TO_EVALUATE = 30

# Random input chỉ là kiểm tra bổ sung.
# Public examples vẫn là điều kiện bắt buộc.
RANDOM_EQUIVALENCE_CASES = 120

# Với một số task, random input không hợp lệ
# cho chính graph baseline. Các case baseline lỗi
# sẽ bị bỏ qua. Cần tối thiểu số case thành công này.
MIN_RANDOM_SUCCESS_CASES = 5

RANDOM_SEED = 715132


# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

def pip_install(
    package_spec,
    no_deps=False,
):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
    ]

    if no_deps:
        command.append("--no-deps")

    command.append(package_spec)

    subprocess.check_call(command)


requirements = {
    "numpy": "numpy",
    "pandas": "pandas",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
}

for module_name, package_name in requirements.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name}...")
        pip_install(package_name)


try:
    current_onnx_tool_version = (
        importlib.metadata.version(
            "onnx-tool"
        )
    )

except importlib.metadata.PackageNotFoundError:
    current_onnx_tool_version = None


if current_onnx_tool_version != "1.0.1":
    print("Installing onnx-tool==1.0.1...")

    pip_install(
        "onnx-tool==1.0.1",
        no_deps=True,
    )


importlib.invalidate_caches()

for module_key in list(sys.modules):
    if (
        module_key == "onnx_tool"
        or module_key.startswith("onnx_tool.")
    ):
        del sys.modules[module_key]


import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort
import onnx_tool

from IPython.display import display
from onnx import helper, numpy_helper


print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("ONNX         :", onnx.__version__)
print("ONNX Runtime :", ort.__version__)
print(
    "onnx-tool    :",
    importlib.metadata.version("onnx-tool"),
)


# ============================================================
# 2. HASH / ZIP HELPERS
# ============================================================

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    return sha256_bytes(path.read_bytes())


def write_submission_zip(
    output_path,
    models,
):
    if output_path.exists():
        output_path.unlink()

    with zipfile.ZipFile(
        output_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as zf:
        for filename in sorted(models):
            zf.writestr(
                filename,
                models[filename],
            )


def read_submission_zip(zip_path):
    models = {}

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            if not member.lower().endswith(".onnx"):
                continue

            filename = Path(member).name.lower()

            if filename in models:
                raise RuntimeError(
                    f"Duplicate ONNX basename: {filename}"
                )

            models[filename] = zf.read(member)

    return models


# ============================================================
# 3. LOAD EXACT BASELINE 7151.32
# ============================================================

if not BASELINE_DIR.exists():
    raise FileNotFoundError(
        "Không tìm thấy baseline directory:\n"
        f"{BASELINE_DIR}"
    )


baseline_paths = {}

for path in BASELINE_DIR.rglob("task*.onnx"):
    filename = path.name.lower()

    if filename not in EXPECTED_TASKS:
        continue

    if filename in baseline_paths:
        old_path = baseline_paths[filename]

        if old_path.read_bytes() != path.read_bytes():
            raise RuntimeError(
                f"Duplicate khác nội dung: {filename}\n"
                f" - {old_path}\n"
                f" - {path}"
            )

    else:
        baseline_paths[filename] = path


found_tasks = set(baseline_paths)

if found_tasks != EXPECTED_TASKS:
    missing = sorted(
        EXPECTED_TASKS - found_tasks
    )

    extra = sorted(
        found_tasks - EXPECTED_TASKS
    )

    raise AssertionError(
        "Baseline không chứa đúng 400 task.\n"
        f"Found   : {len(found_tasks)}\n"
        f"Missing : {missing[:20]}\n"
        f"Extra   : {extra[:20]}"
    )


baseline_models = {
    filename: baseline_paths[filename].read_bytes()
    for filename in sorted(EXPECTED_TASKS)
}


write_submission_zip(
    CONTROL_ZIP,
    baseline_models,
)

control_check = read_submission_zip(
    CONTROL_ZIP
)

if control_check != baseline_models:
    raise AssertionError(
        "Control ZIP không khớp baseline."
    )


print("\n" + "=" * 80)
print("BASELINE READY")
print("=" * 80)

print("Directory :", BASELINE_DIR)
print("Models    :", len(baseline_models))
print("Excluded  :", sorted(EXCLUDED_TASKS))
print("Control   :", CONTROL_ZIP)
print("SHA256    :", sha256_file(CONTROL_ZIP))


# ============================================================
# 4. LOAD OFFICIAL SCORER
# ============================================================

UTILITY_PATH = (
    COMPETITION_DIR
    / "neurogolf_utils"
    / "neurogolf_utils.py"
)

if not UTILITY_PATH.exists():
    utility_matches = sorted(
        KAGGLE_INPUT.rglob(
            "neurogolf_utils.py"
        ),
        key=lambda path: (
            "competitions/neurogolf-2026"
            not in str(path),
            len(path.parts),
            len(str(path)),
        ),
    )

    if not utility_matches:
        raise FileNotFoundError(
            "Không tìm thấy neurogolf_utils.py."
        )

    UTILITY_PATH = utility_matches[0]


utility_parent = str(UTILITY_PATH.parent)

if utility_parent not in sys.path:
    sys.path.insert(0, utility_parent)


module_spec = (
    importlib.util.spec_from_file_location(
        "official_neurogolf_utils_715132",
        UTILITY_PATH,
    )
)

if (
    module_spec is None
    or module_spec.loader is None
):
    raise ImportError(
        "Không load được official scorer."
    )


neurogolf_utils = (
    importlib.util.module_from_spec(
        module_spec
    )
)

module_spec.loader.exec_module(
    neurogolf_utils
)


if not hasattr(
    neurogolf_utils,
    "score_network",
):
    raise AttributeError(
        "Official utility không có score_network()."
    )


print("\nOfficial scorer:")
print(UTILITY_PATH)


# ============================================================
# 5. GRAPH ANALYSIS
# ============================================================

def get_node_attributes(node):
    return {
        attribute.name:
            helper.get_attribute_value(attribute)

        for attribute in node.attribute
    }


def divisors(number):
    if number <= 0:
        return [1]

    result = set()

    limit = int(math.sqrt(number))

    for value in range(1, limit + 1):
        if number % value == 0:
            result.add(value)
            result.add(number // value)

    return sorted(result)


def find_best_dilation_mapping(
    weight,
    old_dilations,
):
    """
    Giữ nguyên offset thực tế của từng weight:

        old_index * old_dilation
        ==
        new_index * new_dilation

    Đồng thời giữ nguyên effective kernel:

        (old_kernel - 1) * old_dilation
        ==
        (new_kernel - 1) * new_dilation
    """

    if weight.ndim != 4:
        return None

    out_channels = int(weight.shape[0])
    in_channels = int(weight.shape[1])

    old_kernel_h = int(weight.shape[2])
    old_kernel_w = int(weight.shape[3])

    old_dilation_h = int(old_dilations[0])
    old_dilation_w = int(old_dilations[1])

    if old_kernel_h <= 1 and old_kernel_w <= 1:
        return None

    nonzero_indices = np.argwhere(
        weight != 0
    )

    if len(nonzero_indices) == 0:
        return None

    spatial_positions = {
        (
            int(index[2]),
            int(index[3]),
        )
        for index in nonzero_indices
    }

    span_h = (
        (old_kernel_h - 1)
        * old_dilation_h
    )

    span_w = (
        (old_kernel_w - 1)
        * old_dilation_w
    )

    candidate_dilations_h = (
        divisors(span_h)
        if span_h > 0
        else [old_dilation_h]
    )

    candidate_dilations_w = (
        divisors(span_w)
        if span_w > 0
        else [old_dilation_w]
    )

    old_parameter_count = int(
        weight.size
    )

    best = None

    for new_dilation_h in candidate_dilations_h:
        if span_h % new_dilation_h != 0:
            continue

        new_kernel_h = (
            span_h // new_dilation_h + 1
        )

        for new_dilation_w in candidate_dilations_w:
            if span_w % new_dilation_w != 0:
                continue

            new_kernel_w = (
                span_w // new_dilation_w + 1
            )

            new_parameter_count = (
                out_channels
                * in_channels
                * new_kernel_h
                * new_kernel_w
            )

            if new_parameter_count >= old_parameter_count:
                continue

            mapping_valid = True

            for old_y, old_x in spatial_positions:
                actual_y = (
                    old_y * old_dilation_h
                )

                actual_x = (
                    old_x * old_dilation_w
                )

                if actual_y % new_dilation_h != 0:
                    mapping_valid = False
                    break

                if actual_x % new_dilation_w != 0:
                    mapping_valid = False
                    break

                new_y = (
                    actual_y // new_dilation_h
                )

                new_x = (
                    actual_x // new_dilation_w
                )

                if not (
                    0 <= new_y < new_kernel_h
                    and
                    0 <= new_x < new_kernel_w
                ):
                    mapping_valid = False
                    break

            if not mapping_valid:
                continue

            reduction = (
                old_parameter_count
                - new_parameter_count
            )

            candidate = {
                "old_kernel": [
                    old_kernel_h,
                    old_kernel_w,
                ],
                "new_kernel": [
                    new_kernel_h,
                    new_kernel_w,
                ],
                "old_dilations": [
                    old_dilation_h,
                    old_dilation_w,
                ],
                "new_dilations": [
                    new_dilation_h,
                    new_dilation_w,
                ],
                "old_parameters":
                    old_parameter_count,
                "new_parameters":
                    new_parameter_count,
                "parameter_reduction":
                    reduction,
                "nonzero_count":
                    int(len(nonzero_indices)),
            }

            if (
                best is None
                or reduction
                > best["parameter_reduction"]
            ):
                best = candidate

    return best


def analyze_model(model):
    initializer_map = {
        initializer.name: initializer
        for initializer in model.graph.initializer
    }

    input_reference_count = Counter(
        input_name
        for node in model.graph.node
        for input_name in node.input
        if input_name
    )

    transformations = []

    for node_index, node in enumerate(
        model.graph.node
    ):
        if node.op_type != "Conv":
            continue

        if len(node.input) < 2:
            continue

        weight_name = node.input[1]

        if weight_name not in initializer_map:
            continue

        # Không sửa initializer được dùng bởi nhiều node.
        if input_reference_count[weight_name] != 1:
            continue

        weight = numpy_helper.to_array(
            initializer_map[weight_name]
        )

        if weight.ndim != 4:
            continue

        attributes = get_node_attributes(
            node
        )

        auto_pad = attributes.get(
            "auto_pad",
            b"NOTSET",
        )

        if isinstance(auto_pad, bytes):
            auto_pad = auto_pad.decode()

        # Chỉ xử lý Conv thông thường.
        if auto_pad not in ("NOTSET", ""):
            continue

        old_dilations = list(
            attributes.get(
                "dilations",
                [1, 1],
            )
        )

        mapping = find_best_dilation_mapping(
            weight,
            old_dilations,
        )

        if mapping is None:
            continue

        transformations.append({
            "node_index": node_index,
            "node_name": node.name,
            "weight_name": weight_name,
            **mapping,
        })

    estimated_reduction = sum(
        transformation[
            "parameter_reduction"
        ]
        for transformation in transformations
    )

    return {
        "transformations": transformations,
        "estimated_reduction":
            estimated_reduction,
    }


# ============================================================
# 6. SCAN ALL NON-EXCLUDED TASKS
# ============================================================

scan_rows = []
task_plans = {}


for filename in sorted(EXPECTED_TASKS):
    if filename in EXCLUDED_TASKS:
        continue

    model_path = baseline_paths[filename]

    try:
        model = onnx.load(model_path)
        onnx.checker.check_model(model)

        analysis = analyze_model(model)

        if analysis["estimated_reduction"] <= 0:
            continue

        task_plans[filename] = analysis

        scan_rows.append({
            "task": filename,
            "file_bytes":
                model_path.stat().st_size,
            "nodes":
                len(model.graph.node),
            "transformable_convs":
                len(
                    analysis["transformations"]
                ),
            "estimated_param_reduction":
                analysis["estimated_reduction"],
            "transformations":
                json.dumps(
                    analysis["transformations"]
                ),
        })

    except Exception as exc:
        print(
            f"Scan warning {filename}: "
            f"{type(exc).__name__}: {exc}"
        )


scan_df = pd.DataFrame(scan_rows)

if scan_df.empty:
    shutil.copy2(
        CONTROL_ZIP,
        FINAL_ZIP,
    )

    raise RuntimeError(
        "Không tìm thấy sparse Conv candidate "
        "trong baseline 7151.32."
    )


scan_df = (
    scan_df
    .sort_values(
        "estimated_param_reduction",
        ascending=False,
    )
    .reset_index(drop=True)
)

scan_df.to_csv(
    SCAN_CSV,
    index=False,
)


print("\n" + "=" * 80)
print("TOP STRUCTURAL OPPORTUNITIES")
print("=" * 80)

display(
    scan_df.head(
        MAX_TASKS_TO_EVALUATE
    )
)


# ============================================================
# 7. BUILD CANDIDATE
# ============================================================

def build_candidate_model(
    baseline_model,
    transformations,
):
    candidate = copy.deepcopy(
        baseline_model
    )

    initializer_indices = {
        initializer.name: index
        for index, initializer
        in enumerate(
            candidate.graph.initializer
        )
    }

    for transformation in transformations:
        node_index = int(
            transformation["node_index"]
        )

        weight_name = (
            transformation["weight_name"]
        )

        node = candidate.graph.node[
            node_index
        ]

        initializer_index = (
            initializer_indices[weight_name]
        )

        old_initializer = (
            candidate.graph.initializer[
                initializer_index
            ]
        )

        old_weight = numpy_helper.to_array(
            old_initializer
        )

        old_dilation_h = int(
            transformation[
                "old_dilations"
            ][0]
        )

        old_dilation_w = int(
            transformation[
                "old_dilations"
            ][1]
        )

        new_dilation_h = int(
            transformation[
                "new_dilations"
            ][0]
        )

        new_dilation_w = int(
            transformation[
                "new_dilations"
            ][1]
        )

        new_kernel_h = int(
            transformation[
                "new_kernel"
            ][0]
        )

        new_kernel_w = int(
            transformation[
                "new_kernel"
            ][1]
        )

        new_weight = np.zeros(
            (
                old_weight.shape[0],
                old_weight.shape[1],
                new_kernel_h,
                new_kernel_w,
            ),
            dtype=old_weight.dtype,
        )

        nonzero_indices = np.argwhere(
            old_weight != 0
        )

        for index in nonzero_indices:
            out_channel = int(index[0])
            in_channel = int(index[1])
            old_y = int(index[2])
            old_x = int(index[3])

            actual_y = (
                old_y * old_dilation_h
            )

            actual_x = (
                old_x * old_dilation_w
            )

            new_y = (
                actual_y // new_dilation_h
            )

            new_x = (
                actual_x // new_dilation_w
            )

            new_weight[
                out_channel,
                in_channel,
                new_y,
                new_x,
            ] = old_weight[
                out_channel,
                in_channel,
                old_y,
                old_x,
            ]

        new_initializer = (
            numpy_helper.from_array(
                new_weight,
                name=weight_name,
            )
        )

        candidate.graph.initializer[
            initializer_index
        ].CopyFrom(
            new_initializer
        )

        retained_attributes = [
            copy.deepcopy(attribute)
            for attribute in node.attribute
            if attribute.name not in {
                "kernel_shape",
                "dilations",
            }
        ]

        del node.attribute[:]

        node.attribute.extend(
            retained_attributes
        )

        node.attribute.extend([
            helper.make_attribute(
                "kernel_shape",
                [
                    new_kernel_h,
                    new_kernel_w,
                ],
            ),
            helper.make_attribute(
                "dilations",
                [
                    new_dilation_h,
                    new_dilation_w,
                ],
            ),
        ])

    onnx.checker.check_model(candidate)

    inferred = (
        onnx.shape_inference.infer_shapes(
            copy.deepcopy(candidate),
            strict_mode=False,
        )
    )

    onnx.checker.check_model(inferred)

    return candidate


# ============================================================
# 8. RUNTIME HELPERS
# ============================================================

def create_session(model_path):
    return ort.InferenceSession(
        str(model_path),
        providers=["CPUExecutionProvider"],
    )


def encode_input_grid(grid):
    grid = np.asarray(
        grid,
        dtype=np.int64,
    )

    if grid.ndim != 2:
        raise ValueError(
            "ARC grid phải có 2 chiều."
        )

    height, width = grid.shape

    if not (
        1 <= height <= 30
        and 1 <= width <= 30
    ):
        raise ValueError(
            f"Invalid grid shape: {grid.shape}"
        )

    if grid.min() < 0 or grid.max() > 9:
        raise ValueError(
            "ARC colors phải nằm trong 0..9."
        )

    tensor = np.zeros(
        (1, 10, 30, 30),
        dtype=np.float32,
    )

    rows, columns = np.indices(
        (height, width)
    )

    tensor[
        0,
        grid,
        rows,
        columns,
    ] = 1.0

    return tensor


def run_session(
    session,
    input_tensor,
):
    input_name = (
        session.get_inputs()[0].name
    )

    return session.run(
        None,
        {input_name: input_tensor},
    )[0]


def safe_run(
    session,
    input_tensor,
):
    try:
        output = run_session(
            session,
            input_tensor,
        )

        return {
            "ok": True,
            "output": output,
            "error": "",
        }

    except Exception as exc:
        return {
            "ok": False,
            "output": None,
            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        }


def encode_expected_like(
    output,
    grid,
):
    grid = np.asarray(
        grid,
        dtype=np.int64,
    )

    expected = np.zeros_like(output)

    height, width = grid.shape

    rows, columns = np.indices(
        (height, width)
    )

    expected[
        0,
        grid,
        rows,
        columns,
    ] = 1

    return expected


# ============================================================
# 9. RANDOM EQUIVALENCE
# ============================================================

def exact_equivalence(
    baseline_path,
    candidate_path,
    random_cases,
    seed,
):
    baseline_session = create_session(
        baseline_path
    )

    candidate_session = create_session(
        candidate_path
    )

    rng = np.random.default_rng(seed)

    grids = [
        np.zeros(
            (1, 1),
            dtype=np.int64,
        ),
        np.zeros(
            (30, 30),
            dtype=np.int64,
        ),
        np.full(
            (30, 30),
            9,
            dtype=np.int64,
        ),
        (
            np.arange(900)
            .reshape(30, 30)
            % 10
        ).astype(np.int64),
    ]

    for _ in range(random_cases):
        height = int(
            rng.integers(1, 31)
        )

        width = int(
            rng.integers(1, 31)
        )

        grids.append(
            rng.integers(
                0,
                10,
                size=(height, width),
                dtype=np.int64,
            )
        )

    successful_cases = 0
    skipped_baseline_errors = 0

    for case_index, grid in enumerate(grids):
        input_tensor = encode_input_grid(
            grid
        )

        baseline_result = safe_run(
            baseline_session,
            input_tensor,
        )

        # Random input có thể không hợp lệ
        # cho chính graph baseline.
        if not baseline_result["ok"]:
            skipped_baseline_errors += 1
            continue

        candidate_result = safe_run(
            candidate_session,
            input_tensor,
        )

        if not candidate_result["ok"]:
            return {
                "ok": False,
                "successful_cases":
                    successful_cases,
                "skipped_baseline_errors":
                    skipped_baseline_errors,
                "case_index": case_index,
                "difference_count": None,
                "error":
                    (
                        "Baseline chạy được nhưng "
                        "candidate lỗi: "
                        + candidate_result["error"]
                    ),
            }

        successful_cases += 1

        if not np.array_equal(
            baseline_result["output"],
            candidate_result["output"],
        ):
            return {
                "ok": False,
                "successful_cases":
                    successful_cases,
                "skipped_baseline_errors":
                    skipped_baseline_errors,
                "case_index": case_index,
                "difference_count":
                    int(
                        np.count_nonzero(
                            baseline_result["output"]
                            !=
                            candidate_result["output"]
                        )
                    ),
                "error":
                    "Output mismatch",
            }

    enough_successful_cases = (
        successful_cases
        >= MIN_RANDOM_SUCCESS_CASES
    )

    return {
        "ok": enough_successful_cases,
        "successful_cases":
            successful_cases,
        "skipped_baseline_errors":
            skipped_baseline_errors,
        "case_index": None,
        "difference_count": 0,
        "error":
            (
                ""
                if enough_successful_cases
                else
                "Quá ít random cases hợp lệ "
                "cho baseline."
            ),
    }


# ============================================================
# 10. PUBLIC VALIDATION
# ============================================================

def validate_public(
    task_filename,
    baseline_path,
    candidate_path,
):
    task_id = Path(
        task_filename
    ).stem

    task_json_path = (
        COMPETITION_DIR
        / f"{task_id}.json"
    )

    if not task_json_path.exists():
        return {
            "ok": False,
            "passed": 0,
            "total": 0,
            "error":
                (
                    "Không tìm thấy "
                    f"{task_json_path}"
                ),
        }

    task_data = json.loads(
        task_json_path.read_text()
    )

    baseline_session = create_session(
        baseline_path
    )

    candidate_session = create_session(
        candidate_path
    )

    passed = 0
    total = 0

    for subset in (
        "train",
        "test",
        "arc-gen",
    ):
        for pair_index, pair in enumerate(
            task_data.get(subset, [])
        ):
            total += 1

            input_tensor = encode_input_grid(
                pair["input"]
            )

            baseline_result = safe_run(
                baseline_session,
                input_tensor,
            )

            candidate_result = safe_run(
                candidate_session,
                input_tensor,
            )

            if not baseline_result["ok"]:
                return {
                    "ok": False,
                    "passed": passed,
                    "total": total,
                    "error":
                        (
                            "Baseline lỗi public "
                            f"{subset}[{pair_index}]: "
                            + baseline_result["error"]
                        ),
                }

            if not candidate_result["ok"]:
                return {
                    "ok": False,
                    "passed": passed,
                    "total": total,
                    "error":
                        (
                            "Candidate lỗi public "
                            f"{subset}[{pair_index}]: "
                            + candidate_result["error"]
                        ),
                }

            baseline_output = (
                baseline_result["output"]
            )

            candidate_output = (
                candidate_result["output"]
            )

            expected_output = (
                encode_expected_like(
                    candidate_output,
                    pair["output"],
                )
            )

            if not np.array_equal(
                baseline_output,
                candidate_output,
            ):
                return {
                    "ok": False,
                    "passed": passed,
                    "total": total,
                    "error":
                        (
                            "Baseline/candidate "
                            f"khác tại "
                            f"{subset}[{pair_index}]"
                        ),
                }

            if not np.array_equal(
                candidate_output,
                expected_output,
            ):
                return {
                    "ok": False,
                    "passed": passed,
                    "total": total,
                    "error":
                        (
                            "Candidate sai expected "
                            f"tại {subset}"
                            f"[{pair_index}]"
                        ),
                }

            passed += 1

    return {
        "ok": True,
        "passed": passed,
        "total": total,
        "error": "",
    }


# ============================================================
# 11. OFFICIAL SCORER
# ============================================================

def official_score(model_path):
    profile_directory = Path(
        tempfile.mkdtemp(
            prefix="neurogolf_profile_"
        )
    )

    try:
        session_options = (
            ort.SessionOptions()
        )

        session_options.enable_profiling = True

        session_options.profile_file_prefix = str(
            profile_directory / "trace"
        )

        session = ort.InferenceSession(
            str(model_path),
            sess_options=session_options,
            providers=["CPUExecutionProvider"],
        )

        sample_input = np.zeros(
            (1, 10, 30, 30),
            dtype=np.float32,
        )

        sample_input[
            0,
            0,
            :,
            :,
        ] = 1.0

        input_name = (
            session.get_inputs()[0].name
        )

        session.run(
            None,
            {input_name: sample_input},
        )

        trace_path = session.end_profiling()

        raw_result = (
            neurogolf_utils.score_network(
                onnx.load(model_path),
                str(trace_path),
            )
        )

        if (
            not isinstance(
                raw_result,
                (tuple, list),
            )
            or len(raw_result) < 2
        ):
            raise TypeError(
                "Unexpected score result: "
                f"{raw_result!r}"
            )

        memory = float(raw_result[0])
        params = float(raw_result[1])

        cost = memory + params

        points = max(
            1.0,
            25.0
            - math.log(
                max(cost, 1.0)
            ),
        )

        return {
            "ok": True,
            "memory": memory,
            "params": params,
            "cost": cost,
            "points": points,
            "raw": repr(raw_result),
            "error": "",
        }

    except Exception as exc:
        return {
            "ok": False,
            "memory": None,
            "params": None,
            "cost": None,
            "points": None,
            "raw": None,
            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        }

    finally:
        shutil.rmtree(
            profile_directory,
            ignore_errors=True,
        )


# ============================================================
# 12. EVALUATE TOP CANDIDATES
# ============================================================

evaluation_rows = []
candidate_paths = {}

top_tasks = (
    scan_df
    .head(
        MAX_TASKS_TO_EVALUATE
    )["task"]
    .tolist()
)


for evaluation_index, filename in enumerate(
    top_tasks,
    start=1,
):
    print(
        f"\n[{evaluation_index}/"
        f"{len(top_tasks)}] "
        f"Evaluating {filename}..."
    )

    baseline_path = (
        baseline_paths[filename]
    )

    candidate_path = (
        WORK_DIR
        / filename.replace(
            ".onnx",
            "_candidate.onnx",
        )
    )

    try:
        baseline_model = onnx.load(
            baseline_path
        )

        candidate_model = (
            build_candidate_model(
                baseline_model,
                task_plans[
                    filename
                ]["transformations"],
            )
        )

        onnx.save(
            candidate_model,
            candidate_path,
        )

        create_session(candidate_path)

        equivalence_result = (
            exact_equivalence(
                baseline_path,
                candidate_path,
                random_cases=
                    RANDOM_EQUIVALENCE_CASES,
                seed=
                    (
                        RANDOM_SEED
                        + evaluation_index
                    ),
            )
        )

        if not equivalence_result["ok"]:
            evaluation_rows.append({
                "task": filename,
                "correctness_ok": False,
                "official_ok": False,
                "gain": None,
                "status":
                    "random_equivalence_failed",
                "error":
                    json.dumps(
                        equivalence_result
                    ),
            })

            continue

        public_result = validate_public(
            filename,
            baseline_path,
            candidate_path,
        )

        if not public_result["ok"]:
            evaluation_rows.append({
                "task": filename,
                "correctness_ok": False,
                "official_ok": False,
                "gain": None,
                "status":
                    "public_validation_failed",
                "error":
                    public_result["error"],
            })

            continue

        baseline_score = official_score(
            baseline_path
        )

        candidate_score = official_score(
            candidate_path
        )

        official_ok = (
            baseline_score["ok"]
            and candidate_score["ok"]
        )

        if not official_ok:
            evaluation_rows.append({
                "task": filename,
                "correctness_ok": True,
                "official_ok": False,
                "gain": None,
                "status":
                    "official_score_failed",
                "error":
                    (
                        baseline_score["error"]
                        + " | "
                        + candidate_score["error"]
                    ),
            })

            continue

        gain = (
            candidate_score["points"]
            - baseline_score["points"]
        )

        cost_reduction = (
            baseline_score["cost"]
            - candidate_score["cost"]
        )

        candidate_paths[
            filename
        ] = candidate_path

        evaluation_rows.append({
            "task": filename,
            "correctness_ok": True,
            "official_ok": True,

            "baseline_bytes":
                baseline_path.stat().st_size,

            "candidate_bytes":
                candidate_path.stat().st_size,

            "baseline_memory":
                baseline_score["memory"],

            "candidate_memory":
                candidate_score["memory"],

            "baseline_params":
                baseline_score["params"],

            "candidate_params":
                candidate_score["params"],

            "baseline_cost":
                baseline_score["cost"],

            "candidate_cost":
                candidate_score["cost"],

            "cost_reduction":
                cost_reduction,

            "baseline_points":
                baseline_score["points"],

            "candidate_points":
                candidate_score["points"],

            "gain": gain,

            "public_examples":
                public_result["total"],

            "random_successful":
                equivalence_result[
                    "successful_cases"
                ],

            "random_skipped":
                equivalence_result[
                    "skipped_baseline_errors"
                ],

            "status":
                (
                    "better"
                    if gain > 0
                    else "not_better"
                ),

            "error": "",
        })

        print(
            "Cost:",
            baseline_score["cost"],
            "->",
            candidate_score["cost"],
        )

        print("Gain:", gain)

    except Exception as exc:
        evaluation_rows.append({
            "task": filename,
            "correctness_ok": False,
            "official_ok": False,
            "gain": None,
            "status": "exception",
            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        })


evaluation_df = pd.DataFrame(
    evaluation_rows
)

if not evaluation_df.empty:
    evaluation_df = (
        evaluation_df
        .sort_values(
            "gain",
            ascending=False,
            na_position="last",
        )
        .reset_index(drop=True)
    )


evaluation_df.to_csv(
    EVALUATION_CSV,
    index=False,
)


print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

display(evaluation_df)


# ============================================================
# 13. SELECT BEST SAFE CANDIDATE
# ============================================================

if evaluation_df.empty:
    valid_candidates = pd.DataFrame()

else:
    valid_candidates = evaluation_df[
        (
            evaluation_df["correctness_ok"]
            == True
        )
        &
        (
            evaluation_df["official_ok"]
            == True
        )
        &
        (
            evaluation_df["gain"] > 0
        )
    ]


if valid_candidates.empty:
    best_task = None
    best_candidate_path = None
    best_gain = 0.0

    final_models = dict(
        baseline_models
    )

    print(
        "\nKhông tìm thấy candidate an toàn "
        "có official gain > 0."
    )

else:
    best_row = valid_candidates.iloc[0]

    best_task = str(
        best_row["task"]
    )

    best_candidate_path = (
        candidate_paths[best_task]
    )

    best_gain = float(
        best_row["gain"]
    )

    final_models = dict(
        baseline_models
    )

    final_models[
        best_task
    ] = (
        best_candidate_path.read_bytes()
    )

    print("\nBEST TASK:", best_task)
    print("Expected local gain:", best_gain)


# ============================================================
# 14. BUILD TEST AND FINAL ZIP
# ============================================================

write_submission_zip(
    TEST_ZIP,
    final_models,
)

write_submission_zip(
    FINAL_ZIP,
    final_models,
)


test_check = read_submission_zip(
    TEST_ZIP
)

final_check = read_submission_zip(
    FINAL_ZIP
)


if set(final_check) != EXPECTED_TASKS:
    raise AssertionError(
        "Final ZIP không chứa đúng 400 task."
    )


if final_check != test_check:
    raise AssertionError(
        "Final ZIP không khớp test ZIP."
    )


changed_tasks = [
    filename
    for filename in sorted(
        EXPECTED_TASKS
    )
    if (
        final_check[filename]
        != baseline_models[filename]
    )
]


expected_changed_tasks = (
    [best_task]
    if best_task is not None
    else []
)


if changed_tasks != expected_changed_tasks:
    raise AssertionError(
        "Changed tasks không đúng: "
        f"{changed_tasks}"
    )


# Đảm bảo task352 vẫn nguyên vẹn.
if (
    final_check["task352.onnx"]
    != baseline_models["task352.onnx"]
):
    raise AssertionError(
        "task352 đã bị thay đổi ngoài ý muốn."
    )


# ============================================================
# 15. REPORT
# ============================================================

report = {
    "baseline_score_label":
        7151.32,

    "baseline_directory":
        str(BASELINE_DIR),

    "baseline_task_count":
        len(baseline_models),

    "excluded_tasks":
        sorted(EXCLUDED_TASKS),

    "structural_candidates":
        len(scan_df),

    "evaluated_tasks":
        len(evaluation_df),

    "selected_task":
        best_task,

    "expected_local_gain":
        best_gain,

    "changed_tasks":
        changed_tasks,

    "unchanged_tasks":
        400 - len(changed_tasks),

    "control_zip":
        str(CONTROL_ZIP),

    "test_zip":
        str(TEST_ZIP),

    "final_zip":
        str(FINAL_ZIP),

    "control_zip_sha256":
        sha256_file(CONTROL_ZIP),

    "test_zip_sha256":
        sha256_file(TEST_ZIP),

    "final_zip_sha256":
        sha256_file(FINAL_ZIP),

    "scan_csv":
        str(SCAN_CSV),

    "evaluation_csv":
        str(EVALUATION_CSV),
}


REPORT_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    )
)


# ============================================================
# 16. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 80)
print("FINAL SUBMISSION READY")
print("=" * 80)

print("\nSelected task:")
print(
    best_task
    if best_task is not None
    else "None"
)

print("\nChanged tasks:")
print(
    changed_tasks
    if changed_tasks
    else "None"
)

print("\nUnchanged tasks:")
print(
    400 - len(changed_tasks),
    "/ 400",
)

print("\nExpected local gain:")
print(best_gain)

print("\nExpected approximate score:")
print(
    7151.32 + best_gain
)

print("\nCONTROL — exact baseline 7151.32:")
print(CONTROL_ZIP)
print(
    "SHA256:",
    report["control_zip_sha256"],
)

print("\nTEST — selected candidate:")
print(TEST_ZIP)
print(
    "SHA256:",
    report["test_zip_sha256"],
)

print("\nSUBMIT THIS FILE:")
print(FINAL_ZIP)
print(
    "SHA256:",
    report["final_zip_sha256"],
)

print("\nScan results:")
print(SCAN_CSV)

print("\nEvaluation results:")
print(EVALUATION_CSV)

print("\nReport:")
print(REPORT_JSON)

print("=" * 80)

Installing onnxruntime...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.8 MB/s eta 0:00:00
Installing onnx-tool==1.0.1...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.5 MB/s eta 0:00:00
ENVIRONMENT
Python       : 3.12.13
NumPy        : 2.4.6
ONNX         : 1.21.0
ONNX Runtime : 1.27.0
onnx-tool    : 1.0.1

BASELINE READY
Directory : /kaggle/input/models/hoangvux/baselinev1/other/default/16
Models    : 400
Excluded  : []
Control   : /kaggle/working/submission_control_715132.zip
SHA256    : f3d13baee3fbe2b4f12fd2e4208d2252d6eacdd7e1589b70a98a7259044f01d1

Official scorer:
/kaggle/input/competitions/neurogolf-2026/neurogolf_utils/neurogolf_utils.py

TOP STRUCTURAL OPPORTUNITIES


,task,file_bytes,nodes,transformable_convs,estimated_param_reduction,transformations
0,task132.onnx,7628,59,1,300,"[{""node_index"": 7, ""node_name"": ""scalar_conv"",..."
1,task201.onnx,8343,59,1,160,"[{""node_index"": 0, ""node_name"": ""labels_f"", ""w..."
2,task020.onnx,5313,31,2,6,"[{""node_index"": 3, ""node_name"": ""detect_center..."
3,task019.onnx,4246,32,1,5,"[{""node_index"": 25, ""node_name"": ""diag_39"", ""w..."
4,task094.onnx,1623,17,2,4,"[{""node_index"": 4, ""node_name"": ""row_score"", ""..."



[1/5] Evaluating task132.onnx...
Cost: 4389.0 -> 4089.0
Gain: 0.0708009693235887

[2/5] Evaluating task201.onnx...


2026-06-24 18:48:18.114043913 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Where node. Name:'output_labels' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/math/element_wise_ops.h:583 void onnxruntime::BroadcastIterator::Append(ptrdiff_t, ptrdiff_t) axis == 1 || axis == largest was false. Attempting to broadcast an axis by a dimension other than 1. 6 by 7

2026-06-24 18:48:18.118799113 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Gather node. Name:'pattern_reversed' Status Message: indices element out of data bounds, idx=5 must be within the inclusive range [-3,2]
2026-06-24 18:48:18.122661589 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Gather node. Name:'pattern_reversed' Status Message: indices element out of data bounds, idx=5 must be within the inclusive range [-3,2]
2026-06-24 18:48:18.1246139

Cost: 4774.0 -> 4614.0
Gain: 0.034089368165076905

[3/5] Evaluating task020.onnx...
Cost: 1532.0 -> 1526.0
Gain: 0.00392413845613504

[4/5] Evaluating task019.onnx...


2026-06-24 18:48:18.524476876 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Mod node. Name:'src_r_19' Status Message: element_wise_ops.cc:2242 operator() Integer modulo by zero
2026-06-24 18:48:18.532193430 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Mod node. Name:'src_r_19' Status Message: element_wise_ops.cc:2242 operator() Integer modulo by zero
2026-06-24 18:48:18.534448759 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Mod node. Name:'src_r_19' Status Message: element_wise_ops.cc:2242 operator() Integer modulo by zero
2026-06-24 18:48:18.536220504 [E:onnxruntime:, sequential_executor.cc:620 ExecuteKernel] Non-zero status code returned while running Mod node. Name:'src_r_19' Status Message: element_wise_ops.cc:2242 operator() Integer modulo by zero
2026-06-24 18:48:18.537378425 [E:onnxruntime:, sequential_execut

Cost: 3084.0 -> 3079.0
Gain: 0.0016225867587209564

[5/5] Evaluating task094.onnx...
Cost: 2713.0 -> 2709.0
Gain: 0.0014754705738369012

EVALUATION RESULTS


,task,correctness_ok,official_ok,baseline_bytes,candidate_bytes,baseline_memory,candidate_memory,baseline_params,candidate_params,baseline_cost,candidate_cost,cost_reduction,baseline_points,candidate_points,gain,public_examples,random_successful,random_skipped,status,error
0,task132.onnx,True,True,7628,6471,3990.0,3990.0,399.0,99.0,4389.0,4089.0,300.0,16.613143,16.683944,0.070801,267,124,0,better,
1,task201.onnx,True,True,8343,7744,4476.0,4476.0,298.0,138.0,4774.0,4614.0,160.0,16.529060,16.563150,0.034089,266,113,11,better,
2,task020.onnx,True,True,5313,5375,1356.0,1356.0,176.0,170.0,1532.0,1526.0,6.0,17.665671,17.669595,0.003924,266,124,0,better,
3,task019.onnx,True,True,4246,4279,2997.0,2997.0,87.0,82.0,3084.0,3079.0,5.0,16.966017,16.967640,0.001623,267,113,11,better,
4,task094.onnx,True,True,1623,1647,2662.0,2662.0,51.0,47.0,2713.0,2709.0,4.0,17.094190,17.095665,0.001475,265,124,0,better,



BEST TASK: task132.onnx
Expected local gain: 0.0708009693235887

FINAL SUBMISSION READY

Selected task:
task132.onnx

Changed tasks:
['task132.onnx']

Unchanged tasks:
399 / 400

Expected local gain:
0.0708009693235887

Expected approximate score:
7151.390800969323

CONTROL — exact baseline 7151.32:
/kaggle/working/submission_control_715132.zip
SHA256: f3d13baee3fbe2b4f12fd2e4208d2252d6eacdd7e1589b70a98a7259044f01d1

TEST — selected candidate:
/kaggle/working/submission_next_task_test.zip
SHA256: bbae2e5472d4ffabedf61195965e56ee7222170de4791beab303eb1967146ab3

SUBMIT THIS FILE:
/kaggle/working/submission.zip
SHA256: bbae2e5472d4ffabedf61195965e56ee7222170de4791beab303eb1967146ab3

Scan results:
/kaggle/working/next_task_scan_715132.csv

Evaluation results:
/kaggle/working/next_task_evaluation_715132.csv

Report:
/kaggle/working/next_task_report_715132.json
